In [5]:
import numpy as np
import pandas as pd
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ============================================================
# Class Custom Transformer - Clip tại 99th Percentile
# ============================================================
class PercentileCapper(BaseEstimator, TransformerMixin):
    def __init__(self, upper_quantile=0.99):
        self.upper_quantile = upper_quantile
        self.upper_bounds_ = {}  # Lưu lại ngưỡng 99% của từng cột

    def fit(self, X, y=None):
        # Học ngưỡng 99th percentile trên dữ liệu Train
        X_df = pd.DataFrame(X)
        for col in X_df.columns:
            self.upper_bounds_[col] = X_df[col].quantile(self.upper_quantile)
        return self

    def transform(self, X, y=None):
        # Clip giá trị vượt ngưỡng (giống rfm_clean[col].clip(upper=upper))
        X_df = pd.DataFrame(X).copy()
        for col in X_df.columns:
            X_df[col] = X_df[col].clip(upper=self.upper_bounds_[col])
        return X_df.values

# ============================================================
# Đọc dữ liệu gốc
# ============================================================
RFM_PATH = r'F:\2025-2026\HK2\Bigdata\Cuối kì\BigData-Final-Project\Data\rfm_dataset.parquet'
rfm = pd.read_parquet(RFM_PATH)
columns_rfm = ['Recency', 'Frequency', 'Monetary']
X = rfm[columns_rfm]
# ============================================================
# Xây dựng Pipeline (2 bước y hệt code gốc của bạn)
# Bước 1: Clip tại 99th percentile
# Bước 2: StandardScaler
# ============================================================
rfm_pipeline = Pipeline(steps=[
    ('percentile_capper', PercentileCapper(upper_quantile=0.99)),
    ('scaler', StandardScaler())
])
# Fit + Transform toàn bộ Pipeline
rfm_scaled = rfm_pipeline.fit_transform(X)
# Lấy dữ liệu sau bước Clip (trước khi Scale) để dùng cho các bước khác
rfm_clean_array = rfm_pipeline.named_steps['percentile_capper'].transform(X)
rfm_clean = pd.DataFrame(rfm_clean_array, columns=columns_rfm, index=X.index)
# In kết quả kiểm tra
print("✅ Đã xử lý outlier (clip tại 99th percentile)")
print(f"✅ Đã chuẩn hóa dữ liệu bằng StandardScaler")
print(f"   Shape sau khi scale: {rfm_scaled.shape}")
print(f"   Mean sau scale (gần 0): {rfm_scaled.mean(axis=0).round(6)}")
print(f"   Std sau scale (gần 1):  {rfm_scaled.std(axis=0).round(6)}")
# Lưu Pipeline
PIPELINE_SAVE_PATH = r'F:\2025-2026\HK2\Bigdata\Cuối kì\BigData-Final-Project\Data\rfm_pipeline_preprocessor.joblib'
joblib.dump(rfm_pipeline, PIPELINE_SAVE_PATH)
print(f"\n💾 Đã lưu Pipeline vào: {PIPELINE_SAVE_PATH}")



✅ Đã xử lý outlier (clip tại 99th percentile)
✅ Đã chuẩn hóa dữ liệu bằng StandardScaler
   Shape sau khi scale: (96096, 3)
   Mean sau scale (gần 0): [-0.  0. -0.]
   Std sau scale (gần 1):  [1. 1. 1.]

💾 Đã lưu Pipeline vào: F:\2025-2026\HK2\Bigdata\Cuối kì\BigData-Final-Project\Data\rfm_pipeline_preprocessor.joblib
